# 2. Pré-processamento dos textos
Nesta etapa, os campos título, resumo e palavras-chave serão reunidos e normalizados para a classificação pelos eixos da BNCC. Registros editoriais e pré-textuais serão excluídos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import limpar_texto, identificar_registro_nao_cientifico
pasta_processados = raiz / 'dados' / '1_processados'

## 2.1 Leitura da saída anterior

In [ ]:
arquivo_entrada = pasta_processados / '01_artigos_consolidados.csv'
df = pd.read_csv(arquivo_entrada, encoding='utf-8-sig')
print(f'Registros recebidos: {len(df)}')
df.head(10)

## 2.2 Construção do texto de análise
Os valores ausentes serão substituídos por texto vazio apenas na coluna combinada. As colunas originais permanecerão inalteradas.

In [ ]:
campos_textuais = ['titulo', 'resumo', 'palavras_chave']
df[campos_textuais].isna().sum().rename('valores_ausentes').to_frame()

### 2.2.1 Normalização (função 'limpar_texto')
- conversão para letras minúsculas;
- retirada de acentos;
- substituição de pontuação e números por espaços;
- remoção de espaços repetidos.

In [ ]:
# Cria uma coluna com o texto original concatenando título e resumo
df['texto_original'] = (
    df['titulo'].fillna('').astype(str) + ' ' +
    df['resumo'].fillna('').astype(str)
).str.strip()

df['titulo_limpo'] = df['titulo'].apply(limpar_texto)
df['texto_limpo'] = df['texto_original'].apply(limpar_texto)

df[['titulo', 'titulo_limpo', 'texto_limpo']].head(10)

## 2.3 Separação de registros que não são artigos
A decisão considera somente o título. Serão separados elementos pré-textuais, editoriais e páginas de abertura do evento.

In [ ]:
df['motivo_exclusao'] = df['titulo_limpo'].apply(identificar_registro_nao_cientifico)
registros_removidos = df[df['motivo_exclusao'] != ''].copy()
artigos = df[df['motivo_exclusao'] == ''].copy()

registros_removidos[['id_artigo', 'evento', 'ano', 'titulo', 'motivo_exclusao']]

## 2.4 Validação das quantidades

In [ ]:
resumo = pd.DataFrame({
    'indicador': ['registros recebidos', 'artigos mantidos', 'registros separados'],
    'quantidade': [len(df), len(artigos), len(registros_removidos)],
})
assert len(df) == len(artigos) + len(registros_removidos)
assert artigos['id_artigo'].is_unique
resumo

## 2.5 Exportação das saídas

In [ ]:
artigos.to_csv(pasta_processados / '02_artigos_pre_processados.csv', index=False, encoding='utf-8-sig')
registros_removidos.to_csv(pasta_processados / '02_registros_removidos.csv', index=False, encoding='utf-8-sig')

print(f'Artigos mantidos: {len(artigos)}')
print(f'Registros separados: {len(registros_removidos)}')